In [1]:
"""
Equity Portfolio Optimization with Net Zero Objectives
Solving Questions 1 and 2
"""

import numpy as np
import pandas as pd
from scipy.optimize import minimize

# ─────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────

sectors = [
    "Communication Services",
    "Consumer Discretionary",
    "Consumer Staples",
    "Energy",
    "Financials",
    "Health Care",
    "Industrials",
    "Information Technology",
    "Materials",
    "Real Estate",
    "Utilities",
]

# Benchmark weights (%)
b = np.array([8.20, 12.30, 6.90, 3.10, 13.20, 12.60, 10.20, 23.00, 4.50, 2.80, 3.20]) / 100

# Betas
beta = np.array([0.95, 1.05, 0.45, 1.40, 1.15, 0.75, 1.00, 1.20, 1.10, 0.80, 0.70])

# Idiosyncratic volatilities (%)
sigma_tilde = np.array([26.2, 32.9, 21.1, 33.8, 23.1, 25.9, 26.5, 27.1, 30.1, 27.4, 22.8]) / 100

# Carbon intensity Scope 1+2 (tCO2e/$ mn)
CI_sc12 = np.array([24, 54, 47, 434, 19, 21, 105, 23, 559, 89, 1655])

# Carbon intensity Scope 1+2+3 upstream (tCO2e/$ mn)
CI_sc123 = np.array([78, 203, 392, 803, 55, 124, 283, 123, 892, 135, 1867])

# Carbon momentum Scope 1+2 (%)
CM_sc12 = np.array([-2.8, -7.2, -1.8, -1.5, -8.3, -7.8, -8.5, -4.3, -7.1, -2.7, -9.9]) / 100

# Carbon momentum Scope 1+2+3 upstream (%)
CM_sc123 = np.array([-0.8, -1.6, -0.1, -0.2, -1.9, -2.0, -2.5, +2.1, -3.6, -0.8, -6.8]) / 100

# Green intensity (%)
GI = np.array([0.0, 1.5, 0.0, 0.7, 0.0, 0.0, 2.4, 0.2, 0.8, 1.4, 8.4]) / 100

# Market parameters
sigma_m = 0.20   # market volatility
r       = 0.03   # risk-free rate
SR      = 0.25   # Sharpe ratio of any portfolio

n = len(sectors)

# ─────────────────────────────────────────────
# QUESTION 1
# ─────────────────────────────────────────────
print("=" * 70)
print("QUESTION 1")
print("=" * 70)

# ── 1(a) Covariance matrix, correlation matrix, sector volatilities ──────
print("\n─── 1(a): Covariance matrix Σ, correlation matrix ρ, volatilities σ ───\n")

# Under CAPM: Σ_ij = β_i β_j σ_m² + δ_ij σ̃_i²
Sigma = np.outer(beta, beta) * sigma_m**2 + np.diag(sigma_tilde**2)

# Sector volatilities
sigma_vec = np.sqrt(np.diag(Sigma))

# Correlation matrix
D_inv = np.diag(1.0 / sigma_vec)
rho = D_inv @ Sigma @ D_inv

print("Sector volatilities σ_i:")
df_vol = pd.DataFrame({"Sector": sectors, "σ_i (%)": np.round(sigma_vec * 100, 2)})
print(df_vol.to_string(index=False))

print("\nCovariance matrix Σ (first 5×5 shown in %):")
print(np.round(Sigma[:5, :5] * 100, 4))

print("\nCorrelation matrix ρ (first 5×5):")
print(np.round(rho[:5, :5], 4))

# ── 1(b) Benchmark volatility ────────────────────────────────────────────
print("\n─── 1(b): Benchmark volatility σ(b) ───\n")

var_b = b @ Sigma @ b
sigma_b = np.sqrt(var_b)
print(f"σ(b) = {sigma_b*100:.4f}%")
print(f"σ_m  = {sigma_m*100:.4f}%")

# ── 1(c) Weighted average beta β(b) ─────────────────────────────────────
print("\n─── 1(c): Weighted-average beta β(b) ───\n")

beta_b = b @ beta
print(f"β(b) = Σ b_i β_i = {beta_b:.6f}")

# ── 1(d) Implied risk premia and expected returns ────────────────────────
print("\n─── 1(d): Implied risk premia π̃ and expected returns μ̃ ───\n")

# Under CAPM: π̃_i = β_i (μ_m - r)
# The market Sharpe ratio is SR = (μ_m - r) / σ_m
# With SR(w) = 0.25 for any portfolio => μ_m - r = SR * σ_m
mu_m_excess = SR * sigma_m        # market excess return
mu_m = mu_m_excess + r

# CAPM implied risk premium for each sector
pi_tilde = beta * mu_m_excess     # β_i * (μ_m - r)
mu_tilde = r + pi_tilde           # expected return

print(f"Market excess return (μ_m - r) = SR × σ_m = {SR} × {sigma_m} = {mu_m_excess:.4f}")
print(f"μ_m = {mu_m:.4f}\n")

df_mu = pd.DataFrame({
    "Sector": sectors,
    "β_i": beta,
    "π̃_i = β_i(μ_m-r) (%)": np.round(pi_tilde * 100, 4),
    "μ̃_i (%)": np.round(mu_tilde * 100, 4),
})
print(df_mu.to_string(index=False))

# ── 1(e) CI(b), CM(b), GI(b) ─────────────────────────────────────────────
print("\n─── 1(e): Climate metrics of the benchmark ───\n")

CI_b_12  = b @ CI_sc12
CI_b_123 = b @ CI_sc123
CM_b_12  = b @ CM_sc12
CM_b_123 = b @ CM_sc123
GI_b     = b @ GI

print(f"CI(b)  Scope 1+2            = {CI_b_12:.2f}  tCO2e/$ mn")
print(f"CI(b)  Scope 1+2+3 upstream = {CI_b_123:.2f} tCO2e/$ mn")
print(f"CM(b)  Scope 1+2            = {CM_b_12*100:.4f}%")
print(f"CM(b)  Scope 1+2+3 upstream = {CM_b_123*100:.4f}%")
print(f"GI(b)                       = {GI_b*100:.4f}%")


# ─────────────────────────────────────────────
# QUESTION 2
# ─────────────────────────────────────────────
print("\n" + "=" * 70)
print("QUESTION 2")
print("=" * 70)

# ── 2(a) Optimization problem ────────────────────────────────────────────
print("\n─── 2(a): Optimization problem ───\n")
print(
    "Objective: minimize tracking-error volatility\n"
    "  min_{w}  (w - b)ᵀ Σ (w - b)\n"
    "\nSubject to:\n"
    "  (i)  Σ w_i = 1                  (fully invested)\n"
    "  (ii) w_i ≥ 0  for all i         (long only)\n"
    "  (iii) CI(w) = Σ w_i CI_i ≤ CI*(t)   (decarbonization constraint)\n"
    "\nWith  CI*(t) = (1 - 0.30)(1 - 0.07)^t × CI(b)\n"
)

# ── Helper: solve QP via scipy ────────────────────────────────────────────
def solve_te_qp(CI_vec, CI_star, Sigma, b, extra_constraints=None):
    """
    Minimize tracking-error variance (w-b)ᵀ Σ (w-b)
    s.t.  sum(w) = 1,  w >= 0,  CI_vec @ w <= CI_star
    extra_constraints: list of scipy LinearConstraint objects
    """
    n = len(b)

    def objective(w):
        diff = w - b
        return diff @ Sigma @ diff

    def grad(w):
        return 2 * Sigma @ (w - b)

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
    if CI_star is not None:
        constraints.append({"type": "ineq", "fun": lambda w: CI_star - CI_vec @ w})
    if extra_constraints:
        constraints.extend(extra_constraints)

    bounds = [(0, 1)] * n
    w0 = b.copy()

    res = minimize(
        objective, w0, jac=grad,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"ftol": 1e-12, "maxiter": 2000},
    )
    return res.x if res.success else None


def portfolio_stats(w, Sigma, b, CI_vec, CM_vec, GI_vec, CI_b):
    te_var = (w - b) @ Sigma @ (w - b)
    te_vol = np.sqrt(te_var) * 100
    ci     = CI_vec @ w
    cm     = CM_vec @ w * 100
    gi     = GI_vec @ w * 100
    red    = (1 - ci / CI_b) * 100
    return te_vol, ci, cm, gi, red


# ── 2(b) Optimized portfolios – Scope 1+2 ───────────────────────────────
print("\n─── 2(b): Optimized portfolios w*(t) – Scope 1+2 ───\n")

time_steps = [0, 1, 2, 5, 10]
results_12 = []

for t in time_steps:
    CI_star = (1 - 0.30) * (1 - 0.07)**t * CI_b_12
    w_opt = solve_te_qp(CI_sc12, CI_star, Sigma, b)
    te, ci, cm, gi, red = portfolio_stats(
        w_opt, Sigma, b, CI_sc12, CM_sc12, GI, CI_b_12
    )
    results_12.append({
        "t": t,
        "CI*(t)": round(CI_star, 2),
        "TE vol (%)": round(te, 4),
        "CI(w) SC1+2": round(ci, 2),
        "CM(w) SC1+2 (%)": round(cm, 4),
        "GI(w) (%)": round(gi, 4),
        "Reduction R(t) (%)": round(red, 4),
        "weights": w_opt,
    })

df_12 = pd.DataFrame(results_12).drop(columns=["weights"])
print(df_12.to_string(index=False))

print("\nWeights w*(t) – Scope 1+2:")
w_df_12 = pd.DataFrame(
    {f"t={r['t']}": np.round(r["weights"] * 100, 2) for r in results_12},
    index=sectors
)
print(w_df_12.to_string())


# ── 2(c) Scope 1+2+3 upstream ────────────────────────────────────────────
print("\n─── 2(c): Optimized portfolios w*(t) – Scope 1+2+3 upstream ───\n")

results_123 = []

for t in time_steps:
    CI_star = (1 - 0.30) * (1 - 0.07)**t * CI_b_123
    w_opt = solve_te_qp(CI_sc123, CI_star, Sigma, b)
    te, ci, cm, gi, red = portfolio_stats(
        w_opt, Sigma, b, CI_sc123, CM_sc123, GI, CI_b_123
    )
    results_123.append({
        "t": t,
        "CI*(t)": round(CI_star, 2),
        "TE vol (%)": round(te, 4),
        "CI(w) SC1+2+3": round(ci, 2),
        "CM(w) SC1+2+3 (%)": round(cm, 4),
        "GI(w) (%)": round(gi, 4),
        "Reduction R(t) (%)": round(red, 4),
        "weights": w_opt,
    })

df_123 = pd.DataFrame(results_123).drop(columns=["weights"])
print(df_123.to_string(index=False))

print("\nWeights w*(t) – Scope 1+2+3:")
w_df_123 = pd.DataFrame(
    {f"t={r['t']}": np.round(r["weights"] * 100, 2) for r in results_123},
    index=sectors
)
print(w_df_123.to_string())


# ── 2(d) Implied bets μ̃_i(w*) - μ̃_i ────────────────────────────────────
print("\n─── 2(d): Implied bets μ̃_i(w*(t)) − μ̃_i ───\n")

# Under the investor's portfolio w*,
# the implied excess returns are π̃_i(w) = [Σw]_i / (w'Σw / SR_p * σ_p)
# More precisely, using the Black framework:
# μ̃(w) = r + λ Σ w   where λ is chosen so SR(w) = 0.25
# λ = SR / σ(w)   and σ(w) = sqrt(w'Σw)

def implied_returns(w, Sigma, r=0.03, SR=0.25):
    sigma_w = np.sqrt(w @ Sigma @ w)
    lam = SR / sigma_w
    return r + lam * (Sigma @ w)

print("── Scope 1+2 ──")
header = f"{'Sector':<30}" + "".join(f"  t={r['t']:>2}" for r in results_12)
print(header)
for i, sec in enumerate(sectors):
    row = f"{sec:<30}"
    for r_dict in results_12:
        w_opt = r_dict["weights"]
        mu_implied = implied_returns(w_opt, Sigma)
        mu_bench   = implied_returns(b, Sigma)
        bet = (mu_implied[i] - mu_bench[i]) * 100
        row += f"  {bet:+6.3f}"
    print(row + "  (%)")

print("\n── Scope 1+2+3 upstream ──")
print(header)
for i, sec in enumerate(sectors):
    row = f"{sec:<30}"
    for r_dict in results_123:
        w_opt = r_dict["weights"]
        mu_implied = implied_returns(w_opt, Sigma)
        mu_bench   = implied_returns(b, Sigma)
        bet = (mu_implied[i] - mu_bench[i]) * 100
        row += f"  {bet:+6.3f}"
    print(row + "  (%)")


# ── 2(e) Expected CI just before t=1 rebalancing ─────────────────────────
print("\n─── 2(e): Expected CI just before t=1 rebalancing ───\n")

# At t=0, we hold w*(0). By t=1, each sector's carbon intensity has evolved
# according to its carbon momentum: CI_i(1) = CI_i(0) * (1 + CM_i)
# So the portfolio CI just before rebalancing at t=1 is:
# CI(w*(0), t=1) = Σ w*_i(0) * CI_i(0) * (1 + CM_i)

w0 = results_12[0]["weights"]   # w*(t=0) under Scope 1+2

CI_evolved = CI_sc12 * (1 + CM_sc12)          # each sector's CI after 1 year
CI_portfolio_evolved = w0 @ CI_evolved

CI_star_t1 = (1 - 0.30) * (1 - 0.07)**1 * CI_b_12

print(f"Portfolio w*(0) CI at t=0  : {w0 @ CI_sc12:.4f} tCO2e/$ mn")
print(f"Expected portfolio CI at t=1 (before rebalancing): {CI_portfolio_evolved:.4f} tCO2e/$ mn")
print(f"CI threshold CI*(t=1)                            : {CI_star_t1:.4f} tCO2e/$ mn")
print(f"Difference (portfolio - threshold)               : {CI_portfolio_evolved - CI_star_t1:.4f} tCO2e/$ mn")

if CI_portfolio_evolved <= CI_star_t1:
    print("\n✓ Portfolio still satisfies the t=1 constraint before rebalancing.")
else:
    print("\n✗ Portfolio violates the t=1 constraint before rebalancing — rebalancing is necessary.")


print("\n" + "=" * 70)
print("ALL COMPUTATIONS COMPLETE")
print("=" * 70)

QUESTION 1

─── 1(a): Covariance matrix Σ, correlation matrix ρ, volatilities σ ───

Sector volatilities σ_i:
                Sector  σ_i (%)
Communication Services    32.36
Consumer Discretionary    39.03
      Consumer Staples    22.94
                Energy    43.89
            Financials    32.60
           Health Care    29.93
           Industrials    33.20
Information Technology    36.20
             Materials    37.28
           Real Estate    31.73
             Utilities    26.76

Covariance matrix Σ (first 5×5 shown in %):
[[10.4744  3.99    1.71    5.32    4.37  ]
 [ 3.99   15.2341  1.89    5.88    4.83  ]
 [ 1.71    1.89    5.2621  2.52    2.07  ]
 [ 5.32    5.88    2.52   19.2644  6.44  ]
 [ 4.37    4.83    2.07    6.44   10.6261]]

Correlation matrix ρ (first 5×5):
[[1.     0.3159 0.2303 0.3745 0.4142]
 [0.3159 1.     0.2111 0.3432 0.3796]
 [0.2303 0.2111 1.     0.2503 0.2768]
 [0.3745 0.3432 0.2503 1.     0.4501]
 [0.4142 0.3796 0.2768 0.4501 1.    ]]

─── 1(b): Benchmar

In [2]:
"""
Equity Portfolio Optimization with Net Zero Objectives
Question 3: Additional Constraints (Scope 1+2 only, t=0)
"""

import numpy as np
import pandas as pd
from scipy.optimize import minimize

# ─────────────────────────────────────────────
# DATA (same as Q1/Q2)
# ─────────────────────────────────────────────

sectors = [
    "Communication Services",
    "Consumer Discretionary",
    "Consumer Staples",
    "Energy",
    "Financials",
    "Health Care",
    "Industrials",
    "Information Technology",
    "Materials",
    "Real Estate",
    "Utilities",
]

b         = np.array([8.20, 12.30, 6.90, 3.10, 13.20, 12.60, 10.20, 23.00, 4.50, 2.80, 3.20]) / 100
beta      = np.array([0.95, 1.05, 0.45, 1.40, 1.15, 0.75, 1.00, 1.20, 1.10, 0.80, 0.70])
sigma_tilde = np.array([26.2, 32.9, 21.1, 33.8, 23.1, 25.9, 26.5, 27.1, 30.1, 27.4, 22.8]) / 100
CI_sc12   = np.array([24, 54, 47, 434, 19, 21, 105, 23, 559, 89, 1655])
CM_sc12   = np.array([-2.8, -7.2, -1.8, -1.5, -8.3, -7.8, -8.5, -4.3, -7.1, -2.7, -9.9]) / 100
GI        = np.array([0.0, 1.5, 0.0, 0.7, 0.0, 0.0, 2.4, 0.2, 0.8, 1.4, 8.4]) / 100

sigma_m = 0.20
n = len(sectors)

# Covariance matrix
Sigma = np.outer(beta, beta) * sigma_m**2 + np.diag(sigma_tilde**2)

# Benchmark climate metrics
CI_b  = b @ CI_sc12
CM_b  = b @ CM_sc12
GI_b  = b @ GI

# CTB threshold at t=0
CI_star_t0 = (1 - 0.30) * CI_b   # 88.95

# ─────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────

def solve_qp(extra_constraints, bounds=None):
    """Minimize TE variance subject to sum=1, long-only, CTB(t=0), + extras."""
    if bounds is None:
        bounds = [(0, 1)] * n

    constraints = [
        {"type": "eq",  "fun": lambda w: np.sum(w) - 1},
        {"type": "ineq","fun": lambda w: CI_star_t0 - CI_sc12 @ w},
    ]
    constraints.extend(extra_constraints)

    def obj(w): return (w - b) @ Sigma @ (w - b)
    def grad(w): return 2 * Sigma @ (w - b)

    res = minimize(obj, b.copy(), jac=grad, method="SLSQP",
                   bounds=bounds, constraints=constraints,
                   options={"ftol": 1e-13, "maxiter": 3000})
    return res.x if res.success else None


def stats(w):
    """Return a dict of portfolio statistics."""
    if w is None:
        return {k: np.nan for k in ["TE vol (%)", "CI", "CM (%)", "GI (%)", "R (%)"]}
    te  = np.sqrt((w - b) @ Sigma @ (w - b)) * 100
    ci  = CI_sc12 @ w
    cm  = CM_sc12 @ w * 100
    gi  = GI @ w * 100
    red = (1 - ci / CI_b) * 100
    return {"TE vol (%)": round(te, 4), "CI": round(ci, 2),
            "CM (%)": round(cm, 4), "GI (%)": round(gi, 4), "R (%)": round(red, 4)}


def print_weights(w, label=""):
    df = pd.DataFrame({"Sector": sectors, f"w* {label} (%)": np.round(w * 100, 2)})
    print(df.to_string(index=False))


# ─────────────────────────────────────────────
# BASELINE: w*(t=0) from Q2(b) for reference
# ─────────────────────────────────────────────
w_base = solve_qp([])   # Q2(b) result

print("=" * 70)
print("QUESTION 3  –  Additional constraints (Scope 1+2, t = 0)")
print("=" * 70)
print(f"\nReference values: CI*(0) = {CI_star_t0:.2f}  |  CI(b) = {CI_b:.2f}")
print(f"                  CM(b)  = {CM_b*100:.4f}%  |  GI(b) = {GI_b*100:.4f}%\n")

# ═══════════════════════════════════════════════════════════════════════
# 3(a)  Minimum-weight constraint  w_i >= b_i / 2
# ═══════════════════════════════════════════════════════════════════════
print("─" * 70)
print("3(a): Minimum-weight constraint  w_i ≥ b_i / 2")
print("─" * 70)

bounds_3a = [(b[i] / 2, 1.0) for i in range(n)]
w_3a = solve_qp([], bounds=bounds_3a)

s = stats(w_3a)
print("Results for w*(t=0) with constraint w_i ≥ b_i/2:\n")
print_weights(w_3a, "3(a)")
print(f"\n  TE vol  = {s['TE vol (%)']:.4f}%")
print(f"  CI(w)   = {s['CI']:.2f}  tCO2e/$ mn")
print(f"  R(t,w)  = {s['R (%)']:.4f}%  (target 30%)")
print(f"  CM(w)   = {s['CM (%)']:.4f}%")
print(f"  GI(w)   = {s['GI (%)']:.4f}%")

# ═══════════════════════════════════════════════════════════════════════
# 3(b)  Carbon-momentum constraint  CM(w) ≤ CM*
# ═══════════════════════════════════════════════════════════════════════
print("─" * 70)
print("3(b): Carbon-momentum constraint  CM(w) ≤ CM*")
print("─" * 70)


cm_targets = [-0.05, -0.06, -0.07, -0.08]
rows_3b = []
weights_3b = {}

for cm_star in cm_targets:
    extra = [{"type": "ineq", "fun": lambda w, c=cm_star: c - CM_sc12 @ w}]
    w_opt = solve_qp(extra)
    s = stats(w_opt)
    s["CM*"] = f"{cm_star*100:.0f}%"
    rows_3b.append(s)
    weights_3b[f"CM*={cm_star*100:.0f}%"] = np.round(w_opt * 100, 2) if w_opt is not None else [np.nan]*n

df_3b = pd.DataFrame(rows_3b)[["CM*", "TE vol (%)", "CI", "R (%)", "CM (%)", "GI (%)"]]
print("\nResults:\n")
print(df_3b.to_string(index=False))

print("\nWeights (%):")
print(pd.DataFrame(weights_3b, index=sectors).to_string())


# ═══════════════════════════════════════════════════════════════════════
# 3(c)  Green-intensity constraint  GI(w) ≥ (1 + G) × GI(b)
# ═══════════════════════════════════════════════════════════════════════
print("─" * 70)
print("3(c): Green-intensity constraint  GI(w) ≥ (1 + G) × GI(b)")
print("─" * 70)


G_values = [0, 0.5, 1, 2]
rows_3c = []
weights_3c = {}

for G in G_values:
    gi_target = (1 + G) * GI_b
    extra = [{"type": "ineq", "fun": lambda w, g=gi_target: GI @ w - g}]
    w_opt = solve_qp(extra)
    s = stats(w_opt)
    s["G"] = G
    s["GI target (%)"] = round(gi_target * 100, 4)
    rows_3c.append(s)
    weights_3c[f"G={G}"] = np.round(w_opt * 100, 2) if w_opt is not None else [np.nan]*n

df_3c = pd.DataFrame(rows_3c)[["G", "GI target (%)", "TE vol (%)", "CI", "R (%)", "CM (%)", "GI (%)"]]
print("\nResults:\n")
print(df_3c.to_string(index=False))

print("\nWeights (%):")
print(pd.DataFrame(weights_3c, index=sectors).to_string())

# ═══════════════════════════════════════════════════════════════════════
# 3(d)  All three constraints combined
# ═══════════════════════════════════════════════════════════════════════
print("─" * 70)
print("3(d): All three constraints combined")
print("─" * 70)
print("""
QP form:
  min  (w - b)ᵀ Σ (w - b)
  s.t. Σ w_i = 1
       CI_sc12 @ w ≤ CI*(0)          (decarbonization)
       w_i ≥ b_i / 2                 (minimum weight)
       CM_sc12 @ w ≤ CM*             (carbon momentum)
       GI @ w ≥ (1 + G) × GI(b)     (green intensity)
       w_i ≤ 1
""")

scenarios = [
    {"CM*": -0.06, "G": 0.25},
    {"CM*":  0.00, "G": 0.50},
    {"CM*": -0.07, "G": 0.00},
    {"CM*": -0.07, "G": 0.25},
]

rows_3d = []
weights_3d = {}

for sc in scenarios:
    cm_star  = sc["CM*"]
    G        = sc["G"]
    gi_target = (1 + G) * GI_b
    label = f"CM*={cm_star*100:.0f}%, G={G}"

    bounds_3d = [(b[i] / 2, 1.0) for i in range(n)]
    extra = [
        {"type": "ineq", "fun": lambda w, c=cm_star:  c - CM_sc12 @ w},
        {"type": "ineq", "fun": lambda w, g=gi_target: GI @ w - g},
    ]
    w_opt = solve_qp(extra, bounds=bounds_3d)
    s = stats(w_opt)
    s["Scenario"] = label
    rows_3d.append(s)
    weights_3d[label] = np.round(w_opt * 100, 2) if w_opt is not None else [np.nan]*n

df_3d = pd.DataFrame(rows_3d)[["Scenario", "TE vol (%)", "CI", "R (%)", "CM (%)", "GI (%)"]]
print("Results:\n")
print(df_3d.to_string(index=False))

print("\nWeights (%):")
print(pd.DataFrame(weights_3d, index=sectors).to_string())

# ═══════════════════════════════════════════════════════════════════════
# 3(e)  Comments and realism assessment
# ═══════════════════════════════════════════════════════════════════════
print("""
─────────────────────────────────────────────────────────────────────
3(e): Comments on all results — which solutions are realistic?
─────────────────────────────────────────────────────────────────────

Summary table (all t=0 Scope 1+2 solutions):
""")

all_rows = []

# Baseline Q2(b)
s = stats(w_base); s["Scenario"] = "Q2(b) baseline (no extra)"; all_rows.append(s)

# 3(a)
s = stats(w_3a); s["Scenario"] = "3(a) w_i >= b_i/2"; all_rows.append(s)

# 3(b) selected
for cm_star in [-0.06, -0.07]:
    extra = [{"type": "ineq", "fun": lambda w, c=cm_star: c - CM_sc12 @ w}]
    w_opt = solve_qp(extra)
    s = stats(w_opt); s["Scenario"] = f"3(b) CM*={cm_star*100:.0f}%"; all_rows.append(s)

# 3(c) selected
for G in [0.5, 1]:
    gi_target = (1 + G) * GI_b
    extra = [{"type": "ineq", "fun": lambda w, g=gi_target: GI @ w - g}]
    w_opt = solve_qp(extra)
    s = stats(w_opt); s["Scenario"] = f"3(c) G={G}"; all_rows.append(s)

# 3(d)
for sc in scenarios:
    cm_star = sc["CM*"]; G = sc["G"]
    gi_target = (1 + G) * GI_b
    bounds_3d = [(b[i] / 2, 1.0) for i in range(n)]
    extra = [
        {"type": "ineq", "fun": lambda w, c=cm_star:  c - CM_sc12 @ w},
        {"type": "ineq", "fun": lambda w, g=gi_target: GI @ w - g},
    ]
    w_opt = solve_qp(extra, bounds=bounds_3d)
    s = stats(w_opt); s["Scenario"] = f"3(d) CM*={cm_star*100:.0f}%, G={G}"; all_rows.append(s)

df_all = pd.DataFrame(all_rows)[["Scenario","TE vol (%)","CI","R (%)","CM (%)","GI (%)"]]
print(df_all.to_string(index=False))


print("=" * 70)
print("QUESTION 3 COMPLETE")
print("=" * 70)

QUESTION 3  –  Additional constraints (Scope 1+2, t = 0)

Reference values: CI*(0) = 88.95  |  CI(b) = 127.07
                  CM(b)  = -5.9322%  |  GI(b) = 0.8410%

──────────────────────────────────────────────────────────────────────
3(a): Minimum-weight constraint  w_i ≥ b_i / 2
──────────────────────────────────────────────────────────────────────
Results for w*(t=0) with constraint w_i ≥ b_i/2:

                Sector  w* 3(a) (%)
Communication Services         8.83
Consumer Discretionary        12.61
      Consumer Staples         7.53
                Energy         2.22
            Financials        14.09
           Health Care        13.22
           Industrials        10.40
Information Technology        23.64
             Materials         2.84
           Real Estate         3.03
             Utilities         1.60

  TE vol  = 0.7977%
  CI(w)   = 88.95  tCO2e/$ mn
  R(t,w)  = 30.0000%  (target 30%)
  CM(w)   = -5.8668%
  GI(w)   = 0.7011%
───────────────────────────────────